# CIFAR-10 + CNN: FGSM & PGD Adversarial Attack (Topic 30)
Run on Colab with GPU: **Runtime -> Change runtime type -> T4 GPU**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import save_image

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

EPOCHS = 10
BATCH_SIZE = 128
LR = 1e-3
MODEL_PATH = "cifar10_cnn.pt"
CLASSES = ["plane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

## Step 1: Model + Data

In [ ]:
class CIFAR10CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 32x32 -> 16x16
        x = self.pool(F.relu(self.conv2(x)))   # 16x16 -> 8x8
        x = self.pool(F.relu(self.conv3(x)))   # 8x8   -> 4x4
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

In [ ]:
# Only ToTensor() -- NO mean/std normalization, so pixels stay in [0, 1].
# This keeps epsilon's meaning consistent with the MNIST experiment.
transform = transforms.ToTensor()
train_set = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_set = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
def train(model, loader, optimizer):
    model.train()
    for epoch in range(1, EPOCHS + 1):
        total_loss = 0.0
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = F.cross_entropy(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * images.size(0)
        print(f"Epoch {epoch}/{EPOCHS} - train loss: {total_loss / len(loader.dataset):.4f}")

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
    return correct / len(loader.dataset)

In [ ]:
model = CIFAR10CNN().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

train(model, train_loader, optimizer)

clean_accuracy = evaluate(model, test_loader)
print(f"\nClean test accuracy: {clean_accuracy * 100:.2f}%")

torch.save(model.state_dict(), MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")

## Step 2: FGSM Attack

In [ ]:
def fgsm_attack(model, images, labels, epsilon):
    images = images.clone().detach().to(DEVICE)
    labels = labels.to(DEVICE)
    images.requires_grad = True

    outputs = model(images)
    loss = F.cross_entropy(outputs, labels)

    model.zero_grad()
    loss.backward()

    grad_sign = images.grad.data.sign()
    adv_images = images + epsilon * grad_sign
    return torch.clamp(adv_images, 0, 1).detach()

@torch.no_grad()
def _accuracy_on(model, images, labels):
    preds = model(images).argmax(dim=1)
    return (preds == labels).sum().item()

def evaluate_fgsm(model, loader, epsilon):
    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        if epsilon == 0.0:
            correct += _accuracy_on(model, images, labels)
        else:
            adv_images = fgsm_attack(model, images, labels, epsilon)
            correct += _accuracy_on(model, adv_images, labels)
        total += labels.size(0)
    return correct / total

In [ ]:
EPSILONS = [0.0, 0.01, 0.02, 0.05, 0.1, 0.15, 0.2]

fgsm_results = {}
print("epsilon | FGSM accuracy")
print("--------|--------------")
for eps in EPSILONS:
    acc = evaluate_fgsm(model, test_loader, eps)
    fgsm_results[eps] = acc
    print(f"{eps:>7.2f} | {acc * 100:12.2f}%")

In [ ]:
def save_sample_images(attack_fn, filename_prefix, epsilon):
    images, labels = next(iter(test_loader))
    images, labels = images[:6].to(DEVICE), labels[:6].to(DEVICE)
    adv_images = attack_fn(model, images, labels, epsilon)

    with torch.no_grad():
        clean_preds = model(images).argmax(dim=1)
        adv_preds = model(adv_images).argmax(dim=1)

    print("\nSample predictions (true -> clean_pred -> adv_pred):")
    for i in range(6):
        true_c, clean_c, adv_c = CLASSES[labels[i]], CLASSES[clean_preds[i]], CLASSES[adv_preds[i]]
        tag = "FOOLED" if adv_preds[i] != labels[i] else "still correct"
        print(f"  true={true_c:6s} clean_pred={clean_c:6s} adv_pred={adv_c:6s}  {tag}")

    save_image(images, f"{filename_prefix}_clean.png", nrow=6)
    save_image(adv_images, f"{filename_prefix}_adversarial.png", nrow=6)
    print(f"Saved {filename_prefix}_clean.png and {filename_prefix}_adversarial.png")

save_sample_images(fgsm_attack, "cifar10_fgsm", epsilon=0.05)

## Step 3: PGD Attack

In [ ]:
PGD_STEPS = 20
PGD_ALPHA = 0.005

def pgd_attack(model, images, labels, epsilon, alpha=PGD_ALPHA, num_steps=PGD_STEPS):
    images = images.clone().detach().to(DEVICE)
    labels = labels.to(DEVICE)
    adv_images = images.clone().detach()

    for _ in range(num_steps):
        adv_images.requires_grad = True
        outputs = model(adv_images)
        loss = F.cross_entropy(outputs, labels)

        model.zero_grad()
        loss.backward()

        grad_sign = adv_images.grad.data.sign()
        adv_images = adv_images.detach() + alpha * grad_sign

        perturbation = torch.clamp(adv_images - images, -epsilon, epsilon)
        adv_images = torch.clamp(images + perturbation, 0, 1).detach()

    return adv_images

def evaluate_pgd(model, loader, epsilon):
    if epsilon == 0.0:
        return fgsm_results[0.0]
    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        adv_images = pgd_attack(model, images, labels, epsilon)
        correct += _accuracy_on(model, adv_images, labels)
        total += labels.size(0)
    return correct / total

In [ ]:
results = []  # will hold one row per epsilon, for the CSV

print("epsilon | FGSM accuracy | PGD accuracy")
print("--------|----------------|-------------")
for eps in EPSILONS:
    pgd_acc = evaluate_pgd(model, test_loader, eps)
    fgsm_acc = fgsm_results[eps]
    print(f"{eps:>7.2f} | {fgsm_acc * 100:13.2f}% | {pgd_acc * 100:11.2f}%")
    results.append({
        "epsilon": eps,
        "fgsm_accuracy": round(fgsm_acc * 100, 2),
        "pgd_accuracy": round(pgd_acc * 100, 2),
    })

In [ ]:
save_sample_images(pgd_attack, "cifar10_pgd", epsilon=0.05)

## Save results as CSV (and download it)

In [ ]:
import csv

csv_path = "cifar10_fgsm_pgd_results.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["epsilon", "fgsm_accuracy", "pgd_accuracy"])
    writer.writeheader()
    writer.writerows(results)

print(f"Saved {csv_path}")

# files.download() only works on Colab. Kaggle sometimes lets the import
# succeed (no ImportError) but the actual download fails as a JS error,
# so we detect the platform directly instead of relying on the import.
import os
IS_KAGGLE = os.path.exists("/kaggle")

if not IS_KAGGLE:
    from google.colab import files
    files.download(csv_path)
else:
    print(f"On Kaggle: find {csv_path} in the Output panel (right side) and download it from there.")

## Download the trained model (.pt) to your computer\n\nOn Colab this triggers a browser download. On Kaggle, saved files appear in the **Output** panel (right sidebar) -- download from there instead.

In [ ]:
if not IS_KAGGLE:
    from google.colab import files
    files.download(MODEL_PATH)
else:
    print(f"On Kaggle: find {MODEL_PATH} in the Output panel (right side) and download it from there.")